# PGD Adversarial Training on CIFAR-10

Trains a ResNet-18 with projected gradient descent adversarial training (Madry et al., ICLR 2018), evaluates the
selected checkpoint under a restart-aware protocol, and runs the obfuscated-gradient sanity checks on the result.

## Prerequisites

1. **Accelerator: GPU.** Cell 1 fails on CPU. A k-step PGD attack needs k forward and backward passes per batch plus one
   for the parameter update, so adversarial training costs roughly eight times standard training: one to two hours on a
   T4, and considerably longer than a session allows on CPU.
2. **Three datasets attached.** The toolkit source, archived so that `attacks/`, `defenses/`, `experiments/` and
   `models/` sit at the top level, and carrying `robustness_table.csv` for the naturally trained baseline; CIFAR-10,
   archived so that `cifar-10-batches-py` sits at the top level; and a checkpoints dataset holding
   `resnet18_cifar10_natural.pth`, the 93.43% model, for the transfer check. Set `TRAIN_FROM_SCRATCH = False` and the
   same checkpoints dataset must also hold `resnet18_cifar10_pgd_at.pth` and `adv_training_history.csv`.
3. **Internet**, required only when no CIFAR-10 dataset is attached, in which case cell 3 downloads it. That download
   runs at roughly 60 kB/s from this host and repeats every session, so attaching the data is worth the one-off upload.

## Design notes

**Imported rather than inlined.** The attack and training implementations are imported from the attached source instead
of pasted into cells, so a single version of each exists. A copy inlined into a notebook can diverge from the tested
implementation without anything failing, and an attack that is subtly wrong still returns correctly shaped tensors
inside the perturbation budget.

**What selection is based on.** Checkpoint selection uses robust accuracy on a 5,000-image split held out of the
training set. The test set is untouched until the final evaluation cell. Selecting on the test set would leak test
information into model selection and the reported figure would no longer be held out.

**Restarts are not optional here.** A single random start understates an attack's strength, and the error is negligible
on a naturally trained model whose robust accuracy is already zero and material on a defended one. Every reported PGD
figure below uses ten restarts, keeping the strongest result per sample. The single-restart table is computed as well,
purely so the two can be compared.

**Uncertainty travels with every number.** Each accuracy is written to `robustness_analysis.csv` with a 95% Wilson
interval and its sample count. On 1,000 images the interval near 47% is roughly three points wide, which is larger than
most of the differences anyone will want to read off the table.

**Recovery.** There is no resume path. The history CSV is flushed every epoch and the best checkpoint is rewritten
whenever it improves, both under `/kaggle/working`, so an interrupted session still leaves usable artefacts. Nothing
under `/kaggle/working` survives the session, so save a version and download the output before closing it.

## 1. Environment

In [ ]:
import os
import platform
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device. Set Accelerator to GPU in the notebook settings. This run costs roughly eight times "
        "standard training and is not viable on CPU."
    )

DEVICE = torch.device("cuda")

# cuDNN selects convolution kernels by benchmark timing by default, so two runs of the same cell can differ in the
# last decimals. Fixing the algorithm makes a rerun on this machine reproducible, at a small cost in throughput.
# Bitwise equality across different GPUs is still not guaranteed and is not claimed anywhere in the output.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"python       {platform.python_version()}")
print(f"torch        {torch.__version__}  (cuda {torch.version.cuda})")
print(f"gpu          {torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"cpu count    {os.cpu_count()}")
print(f"cudnn        deterministic={torch.backends.cudnn.deterministic}, benchmark={torch.backends.cudnn.benchmark}")

## 2. Configuration

`SMOKE` runs a two-epoch job on 1,000 training images to prove the path works before committing GPU hours. Run it once,
confirm the verification cell lists every file, then set `SMOKE = False` and run again. The two modes write to separate
directories so a smoke checkpoint of near-random weights can never be mistaken for a trained one.

`TRAIN_FROM_SCRATCH = False` skips training and evaluates a checkpoint from the attached dataset instead. Since
`/kaggle/working` does not survive a session, this is the mode to use when re-running the evaluation against a model
that has already been trained: it costs roughly forty minutes rather than two hours.

`TrainConfig` is the single source of truth for hyperparameters: it validates its own arguments and serialises itself to
JSON beside the results, so there is no second copy to drift.

**On the epoch budget.** Madry and Rice both train for 200 epochs with decays at 100 and 150. Thirty epochs with decays
at the same fractions is a demonstration budget. It lands below published numbers and, as the first full run showed, it
is too short for robust overfitting to appear. If GPU quota allows, 60 epochs gives a more representative curve for
roughly double the time.

**On `num_workers`.** Two is conservative for a four-vCPU Kaggle instance. Raise it to 3 if the GPU appears to be
waiting on data; the Windows default of zero would leave the T4 idle.

In [ ]:
SMOKE = False
TRAIN_FROM_SCRATCH = True

# Recovery switches. Every expensive step writes its result and reuses it on a later run, so an
# interrupted session resumes rather than restarting. FORCE_RECOMPUTE discards those caches.
# FORCE_RETRAIN is separate and deliberately awkward: train() opens the history CSV with "w", so
# re-running the training cell over a finished run destroys two hours of history irrecoverably.
FORCE_RECOMPUTE = False
FORCE_RETRAIN = False

# PGD-20 at ten restarts over all 10,000 test images costs about twenty minutes on a T4 and removes
# the subset-bias question. Drop to 1 for roughly two minutes when quota is short.
FULL_TEST_RESTARTS = 10

WORKING_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
DOWNLOAD_ROOT = Path("/kaggle/temp/data")  # fallback only; scratch, not saved as notebook output
RUN_NAME = "smoke" if SMOKE else "pgd_at_30ep"
OUT_DIR = WORKING_ROOT / "results" / RUN_NAME
CKPT_DIR = WORKING_ROOT / "checkpoints" / RUN_NAME

# Names looked up inside the attached datasets. The natural checkpoint is renamed on upload: the local file is
# "best_resnet18_cifar10 (1).pth", whose sibling without the suffix is a weaker model trained on the wrong stem.
# The defended checkpoint and history names come from the training module in the toolkit cell, so
# there is no second copy of either string to drift.
# Accepted under either name. The local file carries a space and a parenthesis, and its sibling
# without the suffix is a weaker model trained on the wrong stem, so the real protection is the
# clean-accuracy guard in the sanity-check cell rather than the filename.
NATURAL_CKPT_NAMES = ("resnet18_cifar10_natural.pth", "best_resnet18_cifar10 (1).pth")
BASELINE_TABLE_NAME = "robustness_table.csv"

# Passed to TrainConfig; anything absent keeps the reviewed module default.
REAL_OVERRIDES = {
    "epochs": 30,
    "batch_size": 128,
    "num_workers": 2,
    "device": "cuda",
}

SMOKE_OVERRIDES = {
    "epochs": 2,
    "batch_size": 128,
    "attack_steps": 2,
    "val_attack_steps": 2,
    "val_size": 49000,        # leaves 1,000 training images
    "val_eval_samples": 256,
    "lr_milestones": (0.5,),
    "num_workers": 2,
    "device": "cuda",
}

OVERRIDES = SMOKE_OVERRIDES if SMOKE else REAL_OVERRIDES

if SMOKE and not TRAIN_FROM_SCRATCH:
    raise ValueError("SMOKE has nothing to do without training. Set TRAIN_FROM_SCRATCH = True or SMOKE = False.")

CACHE_DIR = OUT_DIR / "cache"

print(f"mode      {'SMOKE' if SMOKE else 'FULL'}, {'training' if TRAIN_FROM_SCRATCH else 'loading a checkpoint'}")
print(f"outputs   {OUT_DIR}")
print(f"ckpts     {CKPT_DIR}")
print(f"cache     {CACHE_DIR}  (recompute={FORCE_RECOMPUTE}, retrain={FORCE_RETRAIN})")

## 3. Data

The toolkit disables torchvision's downloader, because the certificate on the development machine is expired and the
local copy is authoritative there. This cell resolves the data without modifying the reviewed module.

**Attach a CIFAR-10 dataset.** Zip the `cifar-10-batches-py` directory you already have locally and upload it as a
private dataset. The cell finds it and points `CIFAR10_DATA_ROOT` at it directly, with no copy and no download, since
`/kaggle/input` is readable and the loader only reads.

**The fallback exists but is slow.** Downloading from this host runs at roughly 60 kB/s, so 170 MB takes around
45 minutes, and `/kaggle/temp` is scratch, so it would repeat every session. Attaching the dataset makes this cell
instant and removes a dependency on an external host during a paid GPU session.

In [ ]:
from torchvision.datasets import CIFAR10

CIFAR_MARKER = "cifar-10-batches-py"
REQUIRED = ("data_batch_1", "test_batch", "batches.meta")


def cifar_roots(root: Path, max_depth: int = 4) -> list[Path]:
    """Directories D for which D/cifar-10-batches-py holds the real batch files.

    Matching on the batch files rather than on the directory name matters: Kaggle exposes each
    dataset both at /kaggle/input/<slug> and under /kaggle/input/datasets/<owner>/<slug>, so a
    dataset whose slug equals the marker name produces a directory that looks like a hit and is
    one nesting level short. Duplicates are removed by resolved path, since the two exposures
    point at the same data.
    """
    found: list[Path] = []
    seen: set[Path] = set()
    frontier = [(root, 0)]
    while frontier:
        directory, depth = frontier.pop()
        if all((directory / CIFAR_MARKER / name).is_file() for name in REQUIRED):
            resolved = directory.resolve()
            if resolved not in seen:
                seen.add(resolved)
                found.append(directory)
            continue
        if depth >= max_depth:
            continue
        try:
            frontier.extend((child, depth + 1) for child in directory.iterdir() if child.is_dir())
        except PermissionError:
            continue
    return found


DATA_ROOT = None
for candidate in cifar_roots(INPUT_ROOT):
    try:
        # Constructing verifies the MD5 of every batch file against torchvision's canonical values.
        CIFAR10(root=str(candidate), train=False, download=False)
    except RuntimeError as error:
        print(f"rejected {candidate}: {error}")
        continue
    DATA_ROOT = candidate
    print(f"using attached dataset: {DATA_ROOT}")
    break

if DATA_ROOT is None:
    print("No usable CIFAR-10 dataset under /kaggle/input. Directories seen:")
    for directory in sorted(p for p in INPUT_ROOT.glob("*/*") if p.is_dir()):
        print(f"  {directory}")
    print("Falling back to download: roughly 45 minutes at this host's throughput, repeated every "
          "session because /kaggle/temp is scratch.")
    DATA_ROOT = DOWNLOAD_ROOT
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    CIFAR10(root=str(DATA_ROOT), train=True, download=True)
    CIFAR10(root=str(DATA_ROOT), train=False, download=True)

train_size = len(CIFAR10(root=str(DATA_ROOT), train=True, download=False))
test_size = len(CIFAR10(root=str(DATA_ROOT), train=False, download=False))
if (train_size, test_size) != (50_000, 10_000):
    raise RuntimeError(f"Expected 50000/10000 train/test images, found {train_size}/{test_size}.")

os.environ["CIFAR10_DATA_ROOT"] = str(DATA_ROOT)
print(f"CIFAR10_DATA_ROOT = {DATA_ROOT}  ({train_size} train, {test_size} test)")

## 4. Toolkit

The dataset directory is located by searching for `defenses/adversarial_training.py`, which tolerates both a flat
archive and one wrapped in an extra folder. The signature checks and source hashes exist so that an outdated upload
fails in seconds rather than after an hour of training against a stale attack. Record the printed hashes with any result
you report.

Two candidate roots means two copies of the source are attached and there is no principled way to choose between them,
so the cell stops rather than picking one. The earlier version took the first, which would silently train against
whichever copy the directory walk happened to reach first.

`robustness_eval` is imported here rather than beside the evaluation cells. It resolves the checkpoint inside
`load_model` rather than at import time, so nothing is loaded yet, and a renamed symbol fails now instead of after two
hours of training.

`_common.py` anchors its default paths with `parents[2]`, which resolves to a path that does not exist here. That is
harmless: the defaults are consulted only when the environment variables are absent, and `CIFAR10_DATA_ROOT` is set.

In [ ]:
import hashlib
import inspect

MARKER = Path("defenses/adversarial_training.py")
HASHED_SOURCES = (
    "attacks/pgd.py",
    "attacks/fgsm.py",
    "attacks/cw.py",
    "defenses/adversarial_training.py",
    "experiments/robustness_eval.py",
)


def find_roots(root: Path, marker: Path, max_depth: int = 4) -> list[Path]:
    """Directories D under root for which D/marker is a file, deduplicated by resolved path.

    Kaggle may expose a dataset at /kaggle/input/<slug>, under /kaggle/input/datasets/<owner>/<slug>,
    or only the latter, so the depth of an attached dataset is not fixed.
    """
    found: list[Path] = []
    seen: set[Path] = set()
    frontier = [(root, 0)]
    while frontier:
        directory, depth = frontier.pop()
        if (directory / marker).is_file():
            resolved = directory.resolve()
            if resolved not in seen:
                seen.add(resolved)
                found.append(directory)
            continue
        if depth >= max_depth:
            continue
        try:
            frontier.extend((child, depth + 1) for child in directory.iterdir() if child.is_dir())
        except PermissionError:
            continue
    return found


candidates = find_roots(INPUT_ROOT, MARKER)
if not candidates:
    listing = sorted(str(p) for p in INPUT_ROOT.glob("*/*/*") if p.is_dir())
    raise RuntimeError(
        f"No attached dataset contains {MARKER}. Attach the toolkit dataset. "
        f"Directories under /kaggle/input: {listing}"
    )
if len(candidates) > 1:
    raise RuntimeError(
        f"{len(candidates)} attached datasets contain {MARKER}: {[str(c) for c in candidates]}. "
        "Detach all but one so the run is pinned to a known source."
    )
TOOLKIT_ROOT = candidates[0]

sys.path.insert(0, str(TOOLKIT_ROOT))

from attacks.cw import cw
from attacks.fgsm import fgsm
from attacks.pgd import pgd
from defenses.adversarial_training import CHECKPOINT_FILENAME, HISTORY_FILENAME, TrainConfig, train
from experiments._common import build_loader, load_model

# robustness_eval resolves the checkpoint inside load_model rather than at import time, so importing
# it here is safe before any checkpoint exists and a renamed symbol fails in seconds rather than
# after two hours of training.
from experiments.robustness_eval import (
    ALPHA, CW_C, CW_STEPS, EPS, MODEL_NAME, SEED,
    Evaluation, Result, _clean_predictions, _evaluate, _pgd_multi_restart, _write_csv,
)

pgd_params = list(inspect.signature(pgd).parameters)
expected_params = ["model", "images", "labels", "eps", "alpha", "steps", "random_start"]
if pgd_params[:7] != expected_params:
    raise RuntimeError(f"pgd signature changed: expected {expected_params}, found {pgd_params[:7]}")
if "lr_milestones" not in inspect.signature(TrainConfig).parameters:
    raise RuntimeError("TrainConfig has no lr_milestones parameter; the attached source is stale.")
restart_params = list(inspect.signature(_pgd_multi_restart).parameters)
expected_restart = ["model", "images", "labels", "eps", "alpha", "steps", "restarts", "base_seed"]
if restart_params != expected_restart:
    raise RuntimeError(f"_pgd_multi_restart signature changed: expected {expected_restart}, found {restart_params}")

print(f"toolkit   {TOOLKIT_ROOT}")
for relative in HASHED_SOURCES:
    digest = hashlib.sha256((TOOLKIT_ROOT / relative).read_bytes()).hexdigest()[:12]
    print(f"  {digest}  {relative}")

## 5. Resolved configuration

Decay epochs are printed rather than assumed. The milestones are fractions of the run length and Python rounds halves to
even, so at 30 epochs `round(0.75 * 30)` is 22 rather than 23.

In [ ]:
config = TrainConfig(out_dir=OUT_DIR, ckpt_dir=CKPT_DIR, **OVERRIDES)

train_images = 50_000 - config.val_size
steps_per_epoch = -(-train_images // config.batch_size)

print(f"epochs         {config.epochs}")
print(f"lr             {config.lr}, x{config.lr_gamma} at epochs {config.milestone_epochs()}")
print(f"optimiser      SGD momentum={config.momentum} weight_decay={config.weight_decay}")
print(f"threat model   eps={config.eps:.6f} ({round(config.eps * 255)}/255), alpha={config.alpha:.6f}")
print(f"attack         {config.attack_steps} steps training, {config.val_attack_steps} steps validation")
print(f"data           {train_images} train / {config.val_size} held out, {steps_per_epoch} steps per epoch")
print(f"cost           ~{config.attack_steps + 1}x standard training")
print(f"selection      best held-out robust accuracy over {config.val_eval_samples} images; test set untouched")

## 6. Train

Adversarial examples are generated in eval mode, so the attack's forward passes leave BatchNorm's running statistics
untouched and the training-time threat model matches the evaluation harness. The consequence, expected rather than
faulty, is that those statistics are estimated from the adversarial distribution alone and clean accuracy lands well
below the 93.43% a naturally trained model reaches.

**Read `val_robust_acc` as a selection metric, not a robustness estimate.** It uses 10 PGD steps, deliberately weaker
and cheaper than the reported evaluation and consistent across epochs so that checkpoint comparison is free of attack
noise. It is also measured on 1,024 images, so its standard error near 50% is about 1.6 points, and taking a maximum
over 30 such estimates biases the selected value upward by roughly that scale. The number to quote is the independent
test evaluation below, never `best_val_robust_acc`.

**Expected shape at a long budget.** Both accuracies climb slowly, improve sharply just after the first decay, then
`val_robust_acc` peaks within a few epochs and drifts down while `train_adv_acc` keeps rising. That drift is robust
overfitting (Rice, Wong and Kolter, ICML 2020) and is why the loop keeps the best checkpoint rather than the last.

**What 30 epochs actually produced.** No drift. Robust validation accuracy moved between 0.4639 and 0.4961 from the
first decay to the end, a spread of under two standard errors, so the post-decay trajectory was flat within noise and
the selected epoch was close to arbitrary among the last fourteen. The model is still underfit at this budget. The
honest reading is that robust overfitting was not observed rather than absent, and sixty epochs would be needed to
look for it.

**Three failure signatures.** Robust accuracy collapsing toward zero while the loss keeps falling is catastrophic
overfitting, which should not occur with a multi-step adversary and would point at a bug. Both curves flat from the
first epoch points at the learning rate. And `val_robust_acc` sitting at `val_clean_acc` means the attack is not
working rather than that the model is robust.

In [ ]:
import time

import pandas as pd

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def find_file(root: Path, name: str, max_depth: int = 5) -> Path:
    """Locate exactly one file called `name` under `root`, searching to `max_depth`.

    Deliberately duplicates the walking logic of the two resolvers above rather than refactoring
    them: both were wrong twice before reaching their current form, and they are left untouched.
    """
    found: set[Path] = set()
    frontier = [(root, 0)]
    while frontier:
        directory, depth = frontier.pop()
        if (directory / name).is_file():
            found.add((directory / name).resolve())
        if depth >= max_depth:
            continue
        try:
            frontier.extend((child, depth + 1) for child in directory.iterdir() if child.is_dir())
        except PermissionError:
            continue
    if not found:
        raise FileNotFoundError(f"No file named {name} under {root} within depth {max_depth}.")
    if len(found) > 1:
        raise RuntimeError(f"Ambiguous: {len(found)} files named {name} under {root}: {sorted(found)}")
    return found.pop()

def find_optional(root: Path, names: tuple[str, ...]) -> Path | None:
    """First of `names` present under `root`, or None.

    Both callers feed cells that run hours after this one. Resolving them before training turns a
    missing attachment into a line of output at thirty seconds rather than an exception at two and a
    half hours, on a platform where a session restart destroys everything under /kaggle/working.
    """
    for name in names:
        try:
            return find_file(root, name)
        except FileNotFoundError:
            continue
    return None


NATURAL_CKPT = find_optional(INPUT_ROOT, NATURAL_CKPT_NAMES)
BASELINE_TABLE = find_optional(INPUT_ROOT, (BASELINE_TABLE_NAME,))
print(f"natural checkpoint  {NATURAL_CKPT or 'ABSENT, transfer check will be skipped'}")
print(f"baseline table      {BASELINE_TABLE or 'ABSENT, comparison will be skipped'}")
if NATURAL_CKPT is None or BASELINE_TABLE is None:
    print("Attach the missing files now; neither can be added to a running session.\n")
    
planned_checkpoint = CKPT_DIR / CHECKPOINT_FILENAME
planned_history = OUT_DIR / HISTORY_FILENAME

if TRAIN_FROM_SCRATCH and planned_history.is_file() and not FORCE_RETRAIN:
    raise RuntimeError(
        f"{planned_history} already exists. train() opens it with mode 'w', so re-running this cell would "
        "destroy the previous run's history and there is no resume path: the checkpoint holds weights only, "
        "with no optimizer or scheduler state. Set TRAIN_FROM_SCRATCH = False to evaluate the existing "
        "checkpoint, or FORCE_RETRAIN = True if the previous run really is disposable."
    )

if TRAIN_FROM_SCRATCH:
    started = time.perf_counter()
    result = train(config)
    elapsed = time.perf_counter() - started

    checkpoint_path = result.checkpoint_path
    history_path = result.history_path
    best_epoch = result.best_epoch

    print(f"\nwall clock       {elapsed / 60:.1f} min ({elapsed / config.epochs:.0f}s per epoch)")
    print(f"best epoch       {best_epoch} of {config.epochs}")
    print(f"best val robust  {result.best_val_robust_acc:.4f}  (selection metric, not the reported figure)")
elif planned_checkpoint.is_file() and planned_history.is_file():
    # Same session, evaluation cells re-run after a crash further down.
    checkpoint_path, history_path = planned_checkpoint, planned_history
    history_preview = pd.read_csv(history_path)
    best_epoch = int(history_preview.loc[history_preview["val_robust_acc"].idxmax(), "epoch"])
    print(f"resuming from this session's outputs, {len(history_preview)} epochs")
else:
    checkpoint_path = find_file(INPUT_ROOT, CHECKPOINT_FILENAME)
    history_path = find_file(INPUT_ROOT, HISTORY_FILENAME)
    history_preview = pd.read_csv(history_path)
    # train() keeps a checkpoint only on strict improvement, so the first maximum is the selected epoch.
    best_epoch = int(history_preview.loc[history_preview["val_robust_acc"].idxmax(), "epoch"])

print(f"checkpoint       {checkpoint_path}")
print(f"history          {history_path}")
print(f"best epoch       {best_epoch}")

## 7. Trade-off curves

The right panel is a proxy rather than a generalisation gap in the strict sense: `train_adv_acc` is measured under the
7-step training attack and `val_robust_acc` under the 10-step validation attack, so part of the difference is attack
strength rather than generalisation. The axis label says so, since a figure that travels without its caption should not
overstate what it shows.

In [ ]:
import matplotlib.pyplot as plt

history = pd.read_csv(history_path)
figure_path = OUT_DIR / "adv_training_curves.png"

fig, (ax_acc, ax_gap) = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")

ax_acc.plot(history["epoch"], history["train_adv_acc"], label="train (adversarial, 7 steps)", color="tab:orange")
ax_acc.plot(history["epoch"], history["val_clean_acc"], label="validation (clean)", color="tab:blue")
ax_acc.plot(history["epoch"], history["val_robust_acc"], label="validation (robust, 10 steps)", color="tab:green")
for milestone in config.milestone_epochs():
    ax_acc.axvline(milestone, color="grey", linestyle=":", linewidth=1)
ax_acc.axvline(best_epoch, color="tab:red", linestyle="--", linewidth=1, label=f"selected (epoch {best_epoch})")
ax_acc.set_xlabel("epoch")
ax_acc.set_ylabel("accuracy")
ax_acc.set_title("Accuracy and robustness during adversarial training")
ax_acc.legend(loc="lower right", fontsize=9)
ax_acc.grid(alpha=0.3)

ax_gap.plot(history["epoch"], history["train_adv_acc"] - history["val_robust_acc"], color="tab:purple")
ax_gap.axvline(best_epoch, color="tab:red", linestyle="--", linewidth=1)
ax_gap.set_xlabel("epoch")
ax_gap.set_ylabel("train 7-step minus validation 10-step")
ax_gap.set_title("Robust gap proxy (differing attack strengths)")
ax_gap.grid(alpha=0.3)

fig.savefig(figure_path, dpi=150)
plt.show()

print(history.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## 8. Evaluation helpers

Scoring definitions come from `experiments/robustness_eval.py`, and restarts from its own `_pgd_multi_restart`, so the
defended table is comparable to the naturally trained baseline by construction rather than by transcription. Restart
zero is seeded to `SEED + 0`, exactly what the single-restart evaluation already runs, so ten restarts is one restart
plus nine further starts on the same images and the two are paired.

**Each attack runs once.** `collect` returns the per-sample predictions and norms, `aggregate` reduces them to the four
numbers the table holds, and both tables plus the paired test and the distortion distribution are derived from the same
tensors. The previous arrangement ran FGSM, C&W and PGD twice over, at about four minutes of wasted GPU time.
`aggregate` is checked against `_evaluate` on all four fields, so the local reduction cannot drift from the module's
definition of success without the check firing.

**Everything expensive is cached to disk.** A re-run after a crash reloads the per-sample tensors and the tables rather
than recomputing them, which matters most for the twenty-minute full-test row. `FORCE_RECOMPUTE` discards the caches.

**On seeding.** `_pgd_multi_restart` reseeds the global RNG once per restart, which is what makes it reproducible.
`pgd_restarts` wraps it in `torch.random.fork_rng` so that side effect does not escape. Since the seeding happens
inside the call, the wrapper changes no result.

In [ ]:
import math
from collections.abc import Callable
from contextlib import contextmanager
from typing import Iterator

from torch import Tensor

PGD_RESTARTS_STRONG = 10
FULL_TEST_SAMPLES = 10_000
NATURAL_CLEAN_ACC = 0.9343
NATURAL_CLEAN_TOLERANCE = 0.005
NORM_TOLERANCE = 1e-6  # the table prints six decimals, so agreement below this is agreement


@contextmanager
def checkpoint_override(path: Path) -> Iterator[None]:
    # Leaving CIFAR10_RESNET18_CKPT set would make every later load_model call return whichever
    # model was inspected last, which is the kind of contamination that yields a plausible wrong
    # number rather than an error.
    key = "CIFAR10_RESNET18_CKPT"
    previous = os.environ.get(key)
    os.environ[key] = str(path)
    try:
        yield
    finally:
        if previous is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = previous


def pgd_restarts(model: torch.nn.Module, images: Tensor, labels: Tensor, eps: float, alpha: float,
                 steps: int, restarts: int) -> Tensor:
    # _pgd_multi_restart seeds the global RNG to SEED + restart_index before each start, which is
    # what makes it reproducible and also what leaks into every later consumer of the stream.
    # Forking restores the outer state and changes no result, because the seeding happens inside.
    devices = [images.device] if images.device.type == "cuda" else []
    with torch.random.fork_rng(devices=devices):
        return _pgd_multi_restart(model, images, labels, eps, alpha, steps, restarts, SEED)


def collect(scorer: torch.nn.Module, loader: torch.utils.data.DataLoader, device: torch.device,
            attack_fn: Callable[[Tensor, Tensor], Tensor] | None) -> dict[str, Tensor]:
    # attack_fn None evaluates clean inputs. The scorer is always the model under evaluation; a
    # transfer attack differs only in which model its attack_fn crafts against. Norms are kept for
    # every sample rather than only the broken ones, so any masking can be applied afterwards.
    predictions, labels_out, linf, l2 = [], [], [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        adv = images if attack_fn is None else attack_fn(images, labels)
        with torch.no_grad():
            batch_predictions = scorer(adv).argmax(dim=1)
        delta = (adv - images).flatten(1)
        predictions.append(batch_predictions.cpu())
        labels_out.append(labels.cpu())
        linf.append(delta.abs().amax(dim=1).cpu())
        l2.append(delta.norm(dim=1).cpu())
    return {"preds": torch.cat(predictions), "labels": torch.cat(labels_out),
            "linf": torch.cat(linf), "l2": torch.cat(l2)}


def aggregate(measured: dict[str, Tensor], clean_correct: Tensor) -> Result:
    # Mirrors _evaluate: a success is a sample correct before the attack and wrong after it, the
    # rate is over attackable samples, and both norms average over successes only. The check below
    # compares this against _evaluate itself on all four fields.
    preds, labels = measured["preds"], measured["labels"]
    flipped = clean_correct & (preds != labels)
    attackable, broken = int(clean_correct.sum()), int(flipped.sum())
    accuracy = int((preds == labels).sum()) / labels.numel()
    return Result(
        accuracy,
        broken / attackable if attackable else 0.0,
        float(measured["linf"][flipped].mean()) if broken else 0.0,
        float(measured["l2"][flipped].mean()) if broken else 0.0,
    )


def cached_tensors(path: Path, compute: Callable[[], dict[str, Tensor]]) -> dict[str, Tensor]:
    # Reuse a saved measurement when one exists, so a re-run resumes rather than restarts.
    if path.is_file() and not FORCE_RECOMPUTE:
        print(f"{'reusing':>28}: {path.name}")
        return torch.load(path, weights_only=True)
    payload = compute()
    torch.save(payload, path)
    return payload


def wilson_interval(successes: int, total: int, z: float = 1.96) -> tuple[float, float]:
    # Preferred over the normal approximation: it stays inside [0, 1] and remains sensible at the
    # extremes, where several rows of this table sit.
    if total <= 0:
        raise ValueError(f"total must be positive, got {total}")
    if not 0 <= successes <= total:
        raise ValueError(f"successes must lie in [0, {total}], got {successes}")

    proportion = successes / total
    denominator = 1.0 + z**2 / total
    centre = (proportion + z**2 / (2 * total)) / denominator
    half_width = z * math.sqrt(proportion * (1 - proportion) / total + z**2 / (4 * total**2)) / denominator
    return centre - half_width, centre + half_width


def interval_for(accuracy: float, total: int) -> tuple[float, float]:
    # Guards against pairing an accuracy with the wrong sample count, which would silently narrow
    # every interval rather than fail.
    successes = round(accuracy * total)
    if abs(successes - accuracy * total) > 1e-6:
        raise ValueError(f"accuracy {accuracy} is not a multiple of 1/{total}")
    return wilson_interval(successes, total)


def mcnemar_exact(only_first: int, only_second: int) -> float:
    # Only the discordant pairs are informative: under the null that the two attacks are equally
    # strong, each is a fair coin. Exact rather than chi-squared, because the discordant count here
    # is small and that is where the chi-squared approximation is least reliable.
    if only_first < 0 or only_second < 0:
        raise ValueError(f"counts must be non-negative, got {only_first}, {only_second}")
    discordant = only_first + only_second
    if discordant == 0:
        return 1.0
    tail = sum(math.comb(discordant, k) for k in range(min(only_first, only_second) + 1))
    return min(1.0, 2.0 * tail / 2**discordant)


def report(label: str, accuracy: float, total: int) -> tuple[float, float]:
    low, high = interval_for(accuracy, total)
    print(f"{label:>28}: {accuracy:.4f}  95% CI [{low:.4f}, {high:.4f}]  n={total}")
    return low, high


def table_row(attack: str, eps: float, steps: int, result: Result) -> list[str]:
    # The baseline table's schema, so defended and natural tables concatenate cleanly.
    return [
        MODEL_NAME, "pgd_at", attack,
        "" if math.isnan(eps) else f"{eps:.6f}", str(steps),
        f"{result.accuracy:.4f}", f"{result.success_rate:.4f}",
        f"{result.mean_linf:.6f}", f"{result.mean_l2:.6f}",
    ]


def analysis_record(scope: str, attack: str, restarts: int, total: int, result: Result) -> dict[str, object]:
    # Carries the uncertainty and provenance the fixed table schema has no room for.
    low, high = interval_for(result.accuracy, total)
    return {
        "scope": scope, "attack": attack, "restarts": restarts, "n": total,
        "accuracy": result.accuracy, "ci_low": low, "ci_high": high,
        "success_rate": result.success_rate, "mean_linf": result.mean_linf, "mean_l2": result.mean_l2,
        "device": torch.cuda.get_device_name(0), "torch_version": torch.__version__,
        "base_seed": SEED, "cudnn_deterministic": torch.backends.cudnn.deterministic,
    }


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with checkpoint_override(checkpoint_path):
    defended = load_model(DEVICE)

loader = build_loader()
subset_batch = loader.batch_size
clean = cached_tensors(CACHE_DIR / "subset_clean.pt", lambda: collect(defended, loader, DEVICE, None))
labels_flat = clean["labels"]
clean_correct = clean["preds"] == labels_flat
subset_total = labels_flat.numel()
analysis: dict[tuple[str, str, int], dict[str, object]] = {}


def record(entry: dict[str, object]) -> None:
    """Store one analysis row, keyed so a cell re-run replaces rather than duplicates it."""
    analysis[(str(entry["scope"]), str(entry["attack"]), int(entry["restarts"]))] = entry

print(f"{'defended model':>28}: {checkpoint_path.name}")
print(f"{'subset':>28}: {subset_total} images, batch {subset_batch}, "
      f"{int(clean_correct.sum())} correct before any attack")

## 9. Both protocols, one pass per attack

FGSM has no random start and C&W is deterministic at a fixed penalty, so each is measured once and appears in both
tables unchanged. Only the PGD rows differ between one restart and ten.

The C&W row belongs to a different threat model and must be labelled as such wherever it is quoted. It is an L2 attack
with no L-infinity budget, and on the defended model it spends several times 8/255 in L-infinity, so its accuracy is not
comparable to the PGD rows. What it does measure is minimum distortion, which does not saturate at either end and is
arguably the better robustness statistic: the useful comparison is its mean L2 against the naturally trained model's.

In [ ]:
# name -> (steps, eps, restarts, attack)
SPECS: dict[str, tuple[int, float, int, object]] = {
    "none": (0, math.nan, 1, None),
    "fgsm": (1, EPS, 1, lambda x, y: fgsm(defended, x, y, EPS)),
    "pgd-20 x1": (20, EPS, 1, lambda x, y: pgd_restarts(defended, x, y, EPS, ALPHA, 20, 1)),
    "pgd-50 x1": (50, EPS, 1, lambda x, y: pgd_restarts(defended, x, y, EPS, ALPHA, 50, 1)),
    "pgd-20 x10": (20, EPS, PGD_RESTARTS_STRONG,
                   lambda x, y: pgd_restarts(defended, x, y, EPS, ALPHA, 20, PGD_RESTARTS_STRONG)),
    "pgd-50 x10": (50, EPS, PGD_RESTARTS_STRONG,
                   lambda x, y: pgd_restarts(defended, x, y, EPS, ALPHA, 50, PGD_RESTARTS_STRONG)),
    "cw_l2": (CW_STEPS, math.nan, 1, lambda x, y: cw(defended, x, y, c=CW_C, steps=CW_STEPS)),
}
CSV_LABEL = {"pgd-20 x1": "pgd", "pgd-50 x1": "pgd", "pgd-20 x10": "pgd", "pgd-50 x10": "pgd"}

measurements: dict[str, dict[str, Tensor]] = {"none": clean}
for name, (_, _, _, attack_fn) in SPECS.items():
    if name == "none":
        continue
    measurements[name] = cached_tensors(
        CACHE_DIR / f"subset_{name.replace(' ', '_')}.pt",
        lambda fn=attack_fn: collect(defended, loader, DEVICE, fn),
    )

results = {name: aggregate(measured, clean_correct) for name, measured in measurements.items()}

# The local reduction must reproduce the module's on every field, or one of the two has drifted.
reference = _evaluate(
    defended, loader, DEVICE,
    Evaluation("pgd", 20, EPS, lambda x, y: pgd_restarts(defended, x, y, EPS, ALPHA, 20, 1)),
    _clean_predictions(defended, loader, DEVICE),
)
mine = results["pgd-20 x1"]
for field, theirs, ours in (("accuracy", reference.accuracy, mine.accuracy),
                            ("success_rate", reference.success_rate, mine.success_rate),
                            ("mean_linf", reference.mean_linf, mine.mean_linf),
                            ("mean_l2", reference.mean_l2, mine.mean_l2)):
    if abs(theirs - ours) > NORM_TOLERANCE:
        raise RuntimeError(f"aggregate disagrees with _evaluate on {field}: {ours} against {theirs}")
print(f"\n{'aggregate vs _evaluate':>28}: agrees on all four fields\n")

TABLE_ROWS = {
    "single": ("robustness_table_pgd_at.csv", ["none", "fgsm", "pgd-20 x1", "pgd-50 x1", "cw_l2"]),
    "strong": (f"robustness_table_pgd_at_restarts{PGD_RESTARTS_STRONG}.csv",
               ["none", "fgsm", "pgd-20 x10", "pgd-50 x10", "cw_l2"]),
}
# Written unconditionally rather than cached: both tables are derived from the cached measurements
# at no GPU cost, so a separate cache would only create a state where a discarded .pt and a surviving
# .csv disagree with nothing to notice.
tables: dict[str, list[list[str]]] = {}
for kind, (filename, names) in TABLE_ROWS.items():
    tables[kind] = [table_row(CSV_LABEL.get(n, n), SPECS[n][1], SPECS[n][0], results[n]) for n in names]
    _write_csv(tables[kind], OUT_DIR / filename)
    print(f"{kind + ' table':>28}: {filename}")

for name in SPECS:
    report(name, results[name].accuracy, subset_total)
    record(analysis_record("subset", name, SPECS[name][2], subset_total, results[name]))

strong_acc = {"none": results["none"].accuracy, "fgsm": results["fgsm"].accuracy,
              "pgd-20": results["pgd-20 x10"].accuracy, "pgd-50": results["pgd-50 x10"].accuracy,
              "cw_l2": results["cw_l2"].accuracy}
single_acc = {"pgd-20": results["pgd-20 x1"].accuracy, "pgd-50": results["pgd-50 x1"].accuracy}

## 10. Full test set

The 1,000-image subset is the first thousand in dataset order rather than a random sample. Its bias was negligible on
the naturally trained model and 1.2 points on the defended one, so the reported clean and robust figures come from all
10,000 images. The loader uses the subset's batch size, without which the random start would differ and the first
thousand images would not reproduce.

At ten restarts this is the most expensive step in the notebook, about twenty minutes on a T4. A smoke run falls back
to the 1,000-image subset and a single restart, because a pre-flight check that costs twenty minutes is one nobody
runs. Its numbers are meaningless either way.

In [ ]:
full_samples = 1_000 if SMOKE else FULL_TEST_SAMPLES
full_restarts = 1 if SMOKE else FULL_TEST_RESTARTS
if SMOKE:
    print(f"SMOKE: {full_samples} images at {full_restarts} restart, not the full test set\n")

full_loader = build_loader(num_samples=full_samples, batch_size=subset_batch)
full_clean = cached_tensors(CACHE_DIR / f"full_clean_{full_samples}.pt",
                            lambda: collect(defended, full_loader, DEVICE, None))
full_labels = full_clean["labels"]
full_correct = full_clean["preds"] == full_labels
full_total = full_labels.numel()
if full_total != full_samples:
    raise RuntimeError(f"Expected {full_samples} test images, the loader yields {full_total}.")

full_pgd = cached_tensors(
    CACHE_DIR / f"full_pgd20_{full_samples}_x{full_restarts}.pt",
    lambda: collect(defended, full_loader, DEVICE,
                    lambda x, y: pgd_restarts(defended, x, y, EPS, ALPHA, 20, full_restarts)),
)

full_clean_result = aggregate(full_clean, full_correct)
full_pgd_result = aggregate(full_pgd, full_correct)
scope = "smoke_subset" if SMOKE else "full_test"
report(f"clean, {full_total} images", full_clean_result.accuracy, full_total)
print(f"{'':>28}  naturally trained baseline: {NATURAL_CLEAN_ACC}")
report(f"pgd-20 x{full_restarts}, {full_total} images", full_pgd_result.accuracy, full_total)
record((analysis_record(scope, "none", 1, full_total, full_clean_result)))
record((analysis_record(scope, "pgd-20", full_restarts, full_total, full_pgd_result)))


## 11. Per-sample analysis

Three questions the aggregate table cannot answer, all derived from tensors already measured above at no further GPU
cost.

**Did restarts change anything, or is the difference noise?** One restart and ten are scored on the same images, so two
independent confidence intervals is the wrong comparison and McNemar's exact test on the discordant pairs is the right
one. Restart zero is shared, so ten restarts can only add flips through the nine further starts.

**How much distortion does C&W need against a defended model?** Its mean L2 is pulled by a few hard samples, so the
median and quartiles say more about the typical cost. Minimum distortion is the robustness measure that does not
saturate.

**Where do the adversarial predictions go?** Ten FGSM images collapsed onto four classes at 16/255, frog taking four of
them. A concentrated histogram over a thousand images would say the attack has a preferred direction rather than a
per-image one.

In [ ]:
correct_single = measurements["pgd-20 x1"]["preds"] == labels_flat
correct_strong = measurements["pgd-20 x10"]["preds"] == labels_flat
only_single = int((correct_single & ~correct_strong).sum())   # broken only once restarts were added
only_strong = int((correct_strong & ~correct_single).sum())   # broken only without them
p_value = mcnemar_exact(only_single, only_strong)

print(f"pgd-20 accuracy: {float(correct_single.float().mean()):.4f} at 1 restart, "
      f"{float(correct_strong.float().mean()):.4f} at {PGD_RESTARTS_STRONG}")
print(f"discordant pairs: {only_single} broken only with restarts, {only_strong} only without")
print(f"McNemar exact two-sided p = {p_value:.4g}")

# A near-random model has almost nothing correct to break, so both statistics below are undefined
# there. That is the smoke case, and it is worth reporting rather than raising.
cw_measured = measurements["cw_l2"]
cw_flipped = clean_correct & (cw_measured["preds"] != labels_flat)
cw_broken = int(cw_flipped.sum())
cw_norms = cw_measured["l2"][cw_flipped]
cw_linf_mean = float(cw_measured["linf"][cw_flipped].mean()) if cw_broken else float("nan")
cw_median = float("nan")
if cw_broken:
    quartiles = torch.quantile(cw_norms, torch.tensor([0.25, 0.5, 0.75]))
    cw_median = float(quartiles[1])
    print(f"\ncw L2 over {cw_broken} broken samples: mean {float(cw_norms.mean()):.6f}, "
          f"median {cw_median:.6f}, IQR [{float(quartiles[0]):.6f}, {float(quartiles[2]):.6f}], "
          f"max {float(cw_norms.max()):.6f}")
    print(f"cw mean L-inf over the same samples: {cw_linf_mean:.6f} "
          f"({cw_linf_mean / EPS:.1f}x the {round(EPS * 255)}/255 budget, so a different threat model)")
else:
    print("\ncw broke nothing, so its distortion distribution is undefined on this model")

class_names = loader.dataset.dataset.classes
pgd_flipped = clean_correct & (measurements["pgd-20 x10"]["preds"] != labels_flat)
broken = int(pgd_flipped.sum())
if broken:
    counts = torch.bincount(measurements["pgd-20 x10"]["preds"][pgd_flipped], minlength=len(class_names))
    print(f"\nadversarial predictions over {broken} samples broken by pgd-20 x{PGD_RESTARTS_STRONG}")
    for index in torch.argsort(counts, descending=True):
        if counts[index]:
            print(f"  {class_names[index]:>12}: {int(counts[index]):>4}  ({int(counts[index]) / broken:.1%})")
else:
    print("\npgd-20 broke nothing, so there is no class distribution to report")

record(({
    "scope": "subset", "attack": "cw_l2 distortion", "restarts": 1, "n": cw_broken,
    "accuracy": float("nan"), "ci_low": float("nan"), "ci_high": float("nan"),
    "success_rate": float("nan"), "mean_linf": cw_linf_mean,
    "mean_l2": float(cw_norms.mean()) if cw_broken else float("nan"),
    "device": torch.cuda.get_device_name(0), "torch_version": torch.__version__,
    "base_seed": SEED, "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "median_l2": cw_median, "mcnemar_p": p_value,
}))

## 12. Obfuscated-gradient sanity checks

A robustness number without these behind it is not one to report. Athalye, Carlini and Wagner (ICML 2018) identified
five characteristic behaviours of defences that merely make gradients uninformative. Four are checkable here.

**Unbounded attack.** With the entire pixel box available, any attack should reach zero accuracy. Failing this on a
model with real accuracy means the gradient is uninformative rather than the model robust. The check is vacuous on a
near-random model, which is why it fails harmlessly in a smoke run and is diagnostic here.

**Monotonicity.** Accuracy must not increase as the budget grows, since a larger ball contains the smaller one.

**Iterative beats single-step, and more steps do not hurt.** PGD-20 must be at least as strong as FGSM and PGD-50 at
least as strong as PGD-20.

**Black-box must not beat white-box.** Adversarial examples crafted on the naturally trained model and transferred to
the defended one give the attacker strictly less information than a white-box gradient. If transfer succeeds more often,
the white-box number is measuring gradient masking. This is the check the project has never run, and it needs the
natural checkpoint attached. Its clean accuracy is verified first, because the local directory holds a second file with
the same stem that is a weaker model trained on the wrong architecture, and a transfer number crafted on that would look
entirely plausible.

Two further consistency checks are cheap and worth having: the sweep recomputes 8/255 independently of the table and the
two must agree exactly, and ten restarts cannot be weaker than one.

In [ ]:
# Scales of the reported budget rather than absolute fractions of 255, so that the unit-scale point
# is bit-identical to the table's EPS and ALPHA and the agreement check below cannot fail spuriously.
SWEEP_SCALES = (0.25, 0.5, 1.0, 2.0)

sweep: list[tuple[float, float]] = []
for scale in SWEEP_SCALES:
    eps_value, alpha_value = EPS * scale, ALPHA * scale
    spec = Evaluation(
        "pgd",
        20,
        eps_value,
        # Budgets bound as default arguments: Python closures capture by reference, so bare
        # references would read the final loop values for every spec.
        lambda x, y, e=eps_value, a=alpha_value: pgd_restarts(
            defended, x, y, e, a, 20, PGD_RESTARTS_STRONG
        ),
    )
    # Keyed by scale rather than by rounded epsilon: the unit-scale entry is an independent
    # recomputation of the reported pgd-20 row, and a key shared with an earlier run would make
    # the agreement check compare two cached values instead of two computations.
    accuracy = aggregate(
        cached_tensors(CACHE_DIR / f"sweep_scale{scale}.pt",
                       lambda fn=spec.attack_fn: collect(defended, loader, DEVICE, fn)),
        clean_correct,
    ).accuracy
    sweep.append((eps_value, accuracy))
    report(f"pgd-20 at {round(eps_value * 255):>2}/255", accuracy, subset_total)

if NATURAL_CKPT is None:
    transfer_result = None
    print(f"{'transfer from natural':>28}: skipped, no natural checkpoint attached")
else:
    with checkpoint_override(NATURAL_CKPT):
        natural = load_model(DEVICE)

    natural_measured = cached_tensors(
        CACHE_DIR / "subset_natural_clean.pt", lambda m=natural: collect(m, loader, DEVICE, None)
    )
    natural_clean = float((natural_measured["preds"] == natural_measured["labels"]).float().mean())
    if abs(natural_clean - NATURAL_CLEAN_ACC) > NATURAL_CLEAN_TOLERANCE:
        raise RuntimeError(
            f"{NATURAL_CKPT.name} reads back at {natural_clean:.4f} clean accuracy on the subset, expected about "
            f"{NATURAL_CLEAN_ACC}. This is most likely the weaker checkpoint trained on the wrong stem, and a "
            "transfer result crafted on it would be meaningless."
        )
    print(f"{'natural clean, subset':>28}: {natural_clean:.4f}  from {NATURAL_CKPT.name}")

    transfer_result = aggregate(
        cached_tensors(
            CACHE_DIR / "subset_transfer.pt",
            lambda m=natural: collect(defended, loader, DEVICE,
                                      lambda x, y: pgd_restarts(m, x, y, EPS, ALPHA, 20, PGD_RESTARTS_STRONG)),
        ),
        clean_correct,
    )
    report("transfer from natural", transfer_result.accuracy, subset_total)
    record(analysis_record("subset", "pgd-20 transfer", PGD_RESTARTS_STRONG, subset_total, transfer_result))

    # The natural model has done its job. Both lambdas above bound it as a default argument, so the
    # reference here is the last one and the device memory can go back.
    del natural
    torch.cuda.empty_cache()

unbounded_acc = aggregate(
    cached_tensors(CACHE_DIR / "subset_unbounded.pt",
                   lambda: collect(defended, loader, DEVICE, lambda x, y: pgd(defended, x, y, 1.0, 0.1, steps=50))),
    clean_correct,
).accuracy
print(f"{'pgd-50 unbounded':>28}: {unbounded_acc:.4f}")

In [ ]:
accuracies = [accuracy for _, accuracy in sweep]
sweep_at_eps = next(accuracy for eps_value, accuracy in sweep if eps_value == EPS)

checks = {
    "unbounded attack reaches ~0 accuracy": unbounded_acc < 0.05,
    "accuracy is monotone non-increasing in eps": all(a >= b for a, b in zip(accuracies, accuracies[1:])),
    "pgd-20 at least as strong as fgsm": strong_acc["pgd-20"] <= strong_acc["fgsm"] + 1e-9,
    "pgd-50 at least as strong as pgd-20": strong_acc["pgd-50"] <= strong_acc["pgd-20"] + 1e-9,
    # A check that could not run is not a check that passed, so it is reported as skipped below.
    **({"black-box transfer does not beat white-box": transfer_result.accuracy >= strong_acc["pgd-20"] - 1e-9}
       if transfer_result is not None else {}),
    # The scale-1 sweep point recomputes the reported row through a separate cache entry. On a first
    # run that makes it the only demonstration in the project that the deterministic cuDNN setting
    # holds; on a re-run it compares two cached measurements and only proves the caches agree.
    "sweep reproduces the reported pgd-20 row exactly": abs(sweep_at_eps - strong_acc["pgd-20"]) < 1e-12,
    "restarts do not weaken pgd-20": strong_acc["pgd-20"] <= single_acc["pgd-20"] + 1e-9,
    "restarts do not weaken pgd-50": strong_acc["pgd-50"] <= single_acc["pgd-50"] + 1e-9,
    "restarts do not weaken pgd-50": strong_acc["pgd-50"] <= single_acc["pgd-50"] + 1e-9,
    # A McNemar gate on the restart comparison was removed. Restart zero is shared between
    # the two runs, so the ten-restart attack is nested inside the one-restart attack and the
    # reverse discordant count is zero by construction. The two-sided exact test then reduces
    # to p = 2 * 2**-only_single, which cannot clear 0.05 below six discordant pairs, so the
    # check gated on effect size rather than validity. The counts and the p-value are reported
    # in the per-sample cell and written to the analysis CSV; nothing is gated on them.
}


print()
for description, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}]  {description}")
if transfer_result is None:
    print("  [SKIP]  black-box transfer does not beat white-box")
if not all(checks.values()):
    print("\nAt least one check failed. Do not report a robustness number from this run until it is explained.")

analysis_path = OUT_DIR / "robustness_analysis.csv"
pd.DataFrame(list(analysis.values())).to_csv(analysis_path, index=False)
print(f"\nwritten to {analysis_path}")
print(pd.DataFrame(list(analysis.values())).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## 13. Output verification

The naturally trained baseline is read from the attached toolkit dataset rather than transcribed here. A hardcoded
comparison silently goes stale the next time that table is regenerated, and it has been regenerated once already after
a metric definition was corrected.

In [ ]:
expected = [
    checkpoint_path,
    OUT_DIR / TABLE_ROWS["single"][0],
    OUT_DIR / TABLE_ROWS["strong"][0],
    analysis_path,
    figure_path,
]

if TRAIN_FROM_SCRATCH:
    # Written by this run. When loading a checkpoint, both live in the read-only attached dataset instead.
    expected.extend([history_path, OUT_DIR / "adv_training_config.json"])

missing = [path for path in expected if not path.is_file()]
if missing:
    raise RuntimeError(f"Missing expected outputs: {missing}")

for path in expected:
    print(f"{path.stat().st_size / 1024:>10.1f} KB  {path}")

if SMOKE:
    print("\nSMOKE run: these numbers are meaningless. Set SMOKE = False and run again, and delete "
          f"{CACHE_DIR} first so the smoke measurements cannot be mistaken for real ones.")
else:
    if BASELINE_TABLE is None:
        print(f"\nBaseline comparison skipped: no {BASELINE_TABLE_NAME} attached. Regenerate it with "
              "python -m experiments.robustness_eval against the naturally trained checkpoint.")
    else:
        baseline_table = pd.read_csv(BASELINE_TABLE)
        baseline_acc = {
            (f"pgd-{row.steps}" if row.attack == "pgd" else row.attack): float(row.accuracy)
            for row in baseline_table.itertuples()
        }
        print(f"\nbaseline read from {BASELINE_TABLE}")
        print(f"accuracy on the {subset_total}-image test subset, naturally trained -> "
              f"adversarially trained at {PGD_RESTARTS_STRONG} restarts")
        for key, before in baseline_acc.items():
            after = strong_acc.get(key)
            if after is not None:
                print(f"  {key:>7}: {before:.4f} -> {after:.4f}  ({after - before:+.4f})")

    print("\nReport the full-test figures, not the subset ones, and never best_val_robust_acc.")
    print("The cw_l2 row is a different threat model: read its mean_l2, not its accuracy.")

print(f"\npeak GPU memory during evaluation: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
cache_bytes = sum(path.stat().st_size for path in CACHE_DIR.glob("*.pt"))
print(f"cache: {len(list(CACHE_DIR.glob('*.pt')))} measurements, {cache_bytes / 1e6:.1f} MB, under {CACHE_DIR}")
print("Re-running any evaluation cell reuses these. Set FORCE_RECOMPUTE = True to discard them.")